# Finance CA Assistant - Kaggle MVP

This notebook builds the real CA RAG knowledge base from ICAI/CBIC PDFs, creates chunks and retrieval artifacts, then runs a test query.


## 1. Install Dependencies And Locate Package


In [ ]:
# Install runtime dependencies and fetch the latest project code from GitHub.
!pip install -q pypdf numpy requests pyyaml "sentence-transformers>=3.0" "transformers>=4.51" "accelerate>=0.30" huggingface_hub

import shutil
import subprocess
import sys
from pathlib import Path

KAGGLE_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_URL = "https://github.com/Tussharr13/finance-ca-assistant.git"
PACKAGE_ROOT = KAGGLE_ROOT / "finance-ca-assistant"

if PACKAGE_ROOT.exists():
    shutil.rmtree(PACKAGE_ROOT)

subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, str(PACKAGE_ROOT)],
    check=True,
)

if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

from finance_ca_assistant.pipeline import RAGPipeline

print("Using package root:", PACKAGE_ROOT)
print("Import ok:", RAGPipeline.__name__)


## 2. Configure Kaggle Paths


In [ ]:
from pathlib import Path

from finance_ca_assistant.config import AppConfig, ensure_directories

KAGGLE_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
config = AppConfig(
    data_dir=KAGGLE_ROOT / 'data',
    raw_dir=KAGGLE_ROOT / 'data/raw',
    processed_dir=KAGGLE_ROOT / 'data/processed',
    indices_dir=KAGGLE_ROOT / 'data/indices',
    manifest_path=KAGGLE_ROOT / 'data/pdfs_manifest.json',
    clause_index_path=KAGGLE_ROOT / 'data/clause_index.json',
    chunks_path=KAGGLE_ROOT / 'data/processed/chunks.jsonl',
)
ensure_directories(config)
print(config)


## 3. Download PDFs, Process, Chunk, And Save Artifacts

Safe default: reuse `chunks.jsonl` when it exists. A first run builds a bounded corpus; increase the limits or set `REBUILD_KB = True` only for an intentional rebuild. Section 3 performs ingestion only so embeddings are built once in Section 4.


In [ ]:
from finance_ca_assistant.config import DEFAULT_SOURCE_URLS
from finance_ca_assistant.knowledge_base import build_knowledge_base_from_sources

SOURCE_URLS = DEFAULT_SOURCE_URLS
SOURCE_LIMIT = 3
MAX_PAGES_PER_PDF = 30
MAX_CHUNKS_PER_SOURCE = 300
REBUILD_KB = False

result = build_knowledge_base_from_sources(
    config=config,
    source_urls=SOURCE_URLS,
    limit=SOURCE_LIMIT,
    max_pages_per_pdf=MAX_PAGES_PER_PDF,
    max_chunks_per_source=MAX_CHUNKS_PER_SOURCE,
    rebuild=REBUILD_KB,
    build_artifacts=False,
)
print('Reused cached chunks:', result.reused_chunks)
print('PDFs downloaded/ready:', len(result.pdf_paths))
print('Chunks built:', len(result.chunks))
print('Failed sources:', result.failed_sources)
print('Failed PDFs:', result.failed_pdfs)
print('Artifacts:', result.artifact_paths)

if not result.chunks:
    raise ValueError('No chunks were built. Check Kaggle internet and PDF download logs.')


## 4. Build Pipeline And Test Retrieval

Section 4 is staged to isolate Kaggle failures. Run the cells in order:

1. Check runtime and choose safe model flags.
2. Load optional Hugging Face providers with fallback.
3. Build the hybrid retrieval pipeline.
4. Run an AS 1 retrieval smoke test that is covered by the bounded corpus.

Default behavior: HF embeddings run only when a GPU is available; reranker and LLM stay off until retrieval is stable. Turn them on one at a time after this section passes.


In [ ]:
import gc
import os
from collections import defaultdict

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

try:
    import torch
except Exception as error:
    torch = None
    GPU_AVAILABLE = False
    DEVICE = "cpu"
    RUNTIME_NAME = f"CPU only; torch unavailable: {type(error).__name__}: {error}"
else:
    GPU_AVAILABLE = bool(torch.cuda.is_available())
    DEVICE = "cuda" if GPU_AVAILABLE else "cpu"
    RUNTIME_NAME = torch.cuda.get_device_name(0) if GPU_AVAILABLE else "CPU"

print("Runtime:", RUNTIME_NAME)
print("Device:", DEVICE)

# Stable Kaggle defaults. Increase only after this section passes.
MAX_CHUNKS_PER_SOURCE = 300
ENABLE_HF_EMBEDDINGS = GPU_AVAILABLE
ENABLE_HF_RERANKER = False
ENABLE_HF_LLM = False

HF_EMBED_MODEL = "BAAI/bge-small-en-v1.5"
HF_EMBED_DIMENSIONS = 384
HF_RERANK_MODEL = "BAAI/bge-reranker-base"
HF_LLM_MODEL = "Qwen/Qwen3-0.6B"

print("HF embeddings:", ENABLE_HF_EMBEDDINGS, "|", HF_EMBED_MODEL)
print("HF reranker:", ENABLE_HF_RERANKER, "|", HF_RERANK_MODEL)
print("HF LLM:", ENABLE_HF_LLM, "|", HF_LLM_MODEL)


In [ ]:
from finance_ca_assistant.embeddings.embedding_factory import create_embedding_provider
from finance_ca_assistant.generation.llm_factory import create_llm
from finance_ca_assistant.pipeline import RAGPipeline
from finance_ca_assistant.retrieval.reranker_factory import create_reranker
from finance_ca_assistant.retrieval.result_formatter import format_retrieval_results


def clear_runtime_memory():
    gc.collect()
    if torch is not None and GPU_AVAILABLE:
        torch.cuda.empty_cache()


def load_optional_provider(label, enabled, builder):
    if not enabled:
        print(f"{label}: disabled")
        return None
    try:
        clear_runtime_memory()
        provider = builder()
    except Exception as error:
        print(f"{label}: failed; fallback will be used")
        print(f"  {type(error).__name__}: {error}")
        return None
    print(f"{label}: loaded")
    return provider


embedding_provider = load_optional_provider(
    label="Embedding provider",
    enabled=ENABLE_HF_EMBEDDINGS,
    builder=lambda: create_embedding_provider(
        "huggingface",
        model_name=HF_EMBED_MODEL,
        dimensions=HF_EMBED_DIMENSIONS,
        batch_size=32,
        device=DEVICE,
    ),
)

reranker = load_optional_provider(
    label="Reranker",
    enabled=ENABLE_HF_RERANKER,
    builder=lambda: create_reranker(
        "huggingface",
        model_name=HF_RERANK_MODEL,
        device=DEVICE,
    ),
)

llm = load_optional_provider(
    label="LLM",
    enabled=ENABLE_HF_LLM,
    builder=lambda: create_llm(
        "huggingface",
        model_name=HF_LLM_MODEL,
        device_map="auto" if GPU_AVAILABLE else "cpu",
        dtype="auto",
    ),
)


In [ ]:
def balanced_chunk_sample(chunks, max_per_source=None):
    if max_per_source is None:
        return list(chunks)

    grouped = defaultdict(list)
    for chunk in chunks:
        grouped[str(chunk.get("source") or "unknown")].append(chunk)

    selected = []
    for source in sorted(grouped):
        selected.extend(grouped[source][:max_per_source])
    return selected


pipeline_chunks = balanced_chunk_sample(result.chunks, MAX_CHUNKS_PER_SOURCE)
if not pipeline_chunks:
    raise ValueError("No chunks available. Run Section 3 before Section 4.")

print(f"Pipeline chunks: {len(pipeline_chunks)} / {len(result.chunks)}")
print(f"Sources represented: {len({chunk.get('source') for chunk in pipeline_chunks})}")

pipeline = RAGPipeline.from_chunks(
    pipeline_chunks,
    embedding_provider=embedding_provider,
    reranker=reranker,
    llm=llm,
)

print("Pipeline ready")
print("Embedding model:", pipeline.embedding_provider.model_name)
print("Reranker:", type(reranker).__name__ if reranker else "default keyword reranker")
print("LLM:", type(pipeline.answer_generator.llm).__name__)


In [ ]:
question = "What does AS 1 require for disclosure of accounting policies?"
response = pipeline.answer(question, top_k=6)

print("Retrieved sources:")
print(format_retrieval_results(response.retrieved))
print()
print("Answer:")
response.answer


## 5. Agentic CA Consultation

This turn asks for missing client facts before giving salary tax planning advice.


In [ ]:
consultation = pipeline.consult(
    'I am earning salary and want to save tax legally. Act like my CA and guide me.',
    top_k=6,
)
print('Status:', consultation.status)
print()
print('Message:')
print()
print(consultation.message)
print()
print('Retrieved sources:')
print()
print(consultation.retrieved_sources)


## 6. Save Final Artifacts


In [ ]:
final_artifacts = pipeline.save_artifacts(KAGGLE_ROOT / 'finance_ca_assistant_artifacts')
final_artifacts
